# Alignment of HAADF images with SOFIMA

In [155]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from EDX import *
from utils import *
from utils_sofima import *
import numpy as np
import hyperspy.api as hs
import matplotlib.pyplot as plt
import copy
from importlib import reload

from skimage import data, img_as_float, feature
import tensorstore as ts
from scipy.stats import pearsonr
from datetime import datetime
import warnings
import pickle
from datetime import datetime
import gc

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Allignment settings 

In [156]:
num_frames = 100
suffix = '_100frames_align2zero'

# load data
file_path = "../data/EMD/EDXdataset.emd"

#### Load haadf stack

In [154]:
# load and preprocess
EXD_summed_unaligned, haadf_stack, _ = load_EDX(file_path, first_frame=0, last_frame=num_frames, sum_frames=True, haadf_last_frame=False)

print(haadf_stack.shape)
print(haadf_stack.max(),haadf_stack.dtype)

WARNING | RosettaSciIO | The file contains only one spectrum stream (rsciio.emd._emd_velox:590)


(100, 2048, 2048)
65535.0 float64


#### Align based on the first 20 HAADf images - or 100 (for validation)

In [158]:
sof_obj = get_alignment(haadf_stack, 
                  n_align = num_frames,
                  min_peak_ratio=1.1, 
                  min_peak_sharpness=1.1,
                  max_magnitude=0, 
                  max_deviation=0,
                  patch_size = 100,
                  stride = 25,
                  pad_remove = 50,
                  align_to_zero = True)


  0%|          | 0/99 [00:00<?, ?it/s]

  0%|          | 0/99 [00:00<?, ?it/s]

  0%|          | 0/99 [00:00<?, ?it/s]

  0%|          | 0/99 [00:00<?, ?it/s]

#### Apply the alignment 

In [159]:
haadf_stack_aligned = apply_alignment_2D(haadf_stack, sof_obj, 'uint8')

  0%|          | 0/99 [00:00<?, ?it/s]

#### Evaluate alignment performance (PCC between frames, before and after: can add other metrics later)

In [160]:
print(haadf_stack.shape, haadf_stack_aligned.shape)
print(haadf_stack.dtype, haadf_stack_aligned.dtype)

# make sure the stacks have matched dimensions
pad_remove = sof_obj.pad_remove
haadf_stack = np.transpose(haadf_stack,[1,2,0])[pad_remove:2048-pad_remove,pad_remove:2048-pad_remove, :num_frames].astype('uint8')

print(haadf_stack.shape, haadf_stack_aligned.shape)
print(haadf_stack.dtype, haadf_stack_aligned.dtype)

(100, 2048, 2048) (1948, 1948, 100)
float64 uint8
(1948, 1948, 100) (1948, 1948, 100)
uint8 uint8


In [161]:
pcc_before, pcc_after = eval_alignment(haadf_stack, haadf_stack_aligned)
print('Pearson coeffients before and after: ', np.mean(pcc_before), np.mean(pcc_after))

Pearson coeffients before and after:  0.7241205731442077 0.8991672930002682


#### Visualize the alignmed images

In [162]:
# output folder

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
outdir = f"results/alignments/{timestamp+suffix}"
os.makedirs(outdir, exist_ok=True)


for i in range(haadf_stack.shape[2]):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))

    # Right (aligned)
    ax[1].imshow(haadf_stack_aligned[:, :, i], cmap="gray_r")
    ax[1].set_title(f"Chan {i:d} corr with channel 0: {pcc_after[i]:.3f}")
    ax[1].axis("off")

    # Left (unaligned)
    ax[0].imshow(haadf_stack[:, :, i], cmap="gray_r")
    ax[0].set_title(f"Chan {i:d} corr with channel 0: {pcc_before[i]:.3f}")
    ax[0].axis("off")


    # Save figure
    outfile = os.path.join(outdir, f"slice_{i:03d}.png")
    plt.savefig(outfile, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved alignment figures to: {outdir}")


Saved alignment figures to: results/alignments/20260113_175206_100frames_align2zero


#### Save the alignment transformation

In [163]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"results/sofima_transforms/{timestamp}_sof_object{suffix}.pkl", "wb") as f:
    pickle.dump(sof_obj, f)


#### Load the alignment 

In [ ]:
import dill as pickle
with open(f"results/sofima_transforms/20260109_130147_sof_object_100frames.pkl", "rb") as f:
    sof_obj = pickle.load(f)


### Apply to the EDX cube

#### 1) TensorStore the unaligned EDX frames (this takes long)

In [164]:
import logging
logging.getLogger("rsciio.emd").setLevel(logging.ERROR)


In [ ]:
tmp = store_unaligned_hsi_alt(file_path,'tmp/unaligned_hsi'+suffix,n_frames=num_frames)

### Create two EDX objects, one aligned, one not

#### load data from EMD

In [ ]:
edx_unaligned, haadf, xray_energies = load_EDX(file_path, first_frame=0, last_frame=num_frames, sum_frames=True, haadf_last_frame= False)

#### Unaligned

In [ ]:
tile_1 = EM_EDX(haadf[0,:,:], edx_unaligned, xray_energies)
tile_1.apply("crop", parameters={"crop_idx": (slice(None), slice(None), slice(96, 4096))})
tile_1.apply("binning", parameters={"dim": (2048, 2048, 250)})
#tile_1.apply("MeanFilterEDX", parameters={"kernel_size": 3})

#### Aligned

In [ ]:
pad_remove = sof_obj.pad_remove
tile_2 = EM_EDX(haadf[0,:,:], edx_unaligned, xray_energies)
tile_2.apply("crop", parameters={"crop_idx": (slice(None), slice(None), slice(96, 4096))})
tile_2.apply("binning", parameters={"dim": (2048, 2048, 250)})


In [ ]:
# Align
tile_2.apply("sofima_align", parameters={"hsi_stack_path": "tmp/unaligned_hsi"+suffix, "alignment": sof_obj, "data_type": "float32",
                                        "save_aligned": False, "hsi_stack_aligned_path": "tmp/aligned_hsi"+suffix})   


#### Save an aligned tile to memory

In [ ]:
# save to memory
import pickle
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"results/preprocessed_edx/{timestamp}_tile_aligned{suffix}.pkl", "wb") as f:
    pickle.dump(tile_2, f)



#### load the above

In [ ]:
with open('results/preprocessed_edx/20251201_142554_tile_aligned.pkl', 'rb') as file:
    tile_2 = pickle.load(file)

tile_2.summary()

#### Aligned cont.

In [ ]:
tile_2.apply("binning", parameters={"dim": (1024, 1024, 250)})
tile_2.summary()

#### Add meal filter

In [ ]:
tile_3 = tile_1.apply("MeanFilterEDX", parameters={"kernel_size": 3},copy_instance='True')
tile_4 = tile_2.apply("MeanFilterEDX", parameters={"kernel_size": 3},copy_instance='True')

tiles = [tile_1,tile_2,tile_3,tile_4]
for tile in tiles:
    print(tile.summary())

In [ ]:
bands = [24,24,24]
f, ax = plt.subplots(2,2,figsize=(15,15))
#x1,x2,y1,y2 = [750,1000,250,750]
#x1,x2,y1,y2 = [600,1000,400,800]
x1,x2,y1,y2 = [0,1024,0,1024]


#ax[0][0].imshow(tile_1.FalseColor(bands)[x1:x2,y1:y2,:])
ax[0][0].imshow(tile_1.FalseColor(bands)[x1:x2,y1:y2,:])
ax[0][1].imshow(tile_2.FalseColor(bands)[x1:x2,y1:y2,:])


#tmp1 = tile_1.FalseColor(bands)[x1:x2,y1:y2,:]
#tmp2 = tile_2.FalseColor(bands)[x1:x2,y1:y2,:]
#print(np.array_equal(tmp1,tmp2))

ax[1][0].imshow(tile_3.FalseColor(bands)[x1:x2,y1:y2,:])
ax[1][1].imshow(tile_4.FalseColor(bands)[x1:x2,y1:y2,:])

ax[0][0].set_title('Unaligned')
ax[0][1].set_title('aligned')
ax[1][0].set_title('Unaligned+MF')
ax[1][1].set_title('aligned+MF')


## Sillhouette score

In [ ]:
from sklearn.metrics import silhouette_samples
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

# get the hand-annotated masks and display them
mask_dir = os.path.join(os.path.dirname(os.getcwd()),'data', 'masks_frame0_sil_specific')
masks = create_masks(mask_dir)
colors = ['k','r','g','b','c','m','y']
newcmap = ListedColormap(colors)
plt.imshow(masks,cmap=newcmap)
plt.show()

In [ ]:
# taking care of NaN values that remain after binning - what a pain in the back
nanmask = np.isnan(x).all(axis=2)
pad_bin = 0
for r in range(nanmask.shape[0]):
    if nanmask[r, :].all():
        pad_bin += 1
    else:
        break

# compute sil scores for both
sil_img1 = sil_scores(tile_1.EDX[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin,:],
                      masks[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin], metric='euclidean')
sil_img2 = sil_scores(tile_2.EDX[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin,:],
                      masks[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin], metric='euclidean')
sil_img3 = sil_scores(tile_3.EDX[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin,:],
                      masks[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin], metric='euclidean')
sil_img4 = sil_scores(tile_4.EDX[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin,:],
                      masks[pad_bin:1024-pad_bin,pad_bin:1024-pad_bin], metric='euclidean')

# compute global vmin/vmax 
vmin = np.nanmin(np.array([sil_img1,sil_img2,sil_img3,sil_img4]))
vmax = np.nanmax(np.array([sil_img1,sil_img2,sil_img3,sil_img4]))

In [ ]:
# modify jet so NaNs appear black
cmap = plt.cm.jet.copy()
cmap.set_bad(color='black')

bands=[4,25,28]
f, ax = plt.subplots(4,2,figsize=(10, 25))
ax[0][0].imshow(tile_1.FalseColor(bands))
ax[0][0].set_title('Unaligned')
ax[1][0].imshow(tile_2.FalseColor(bands))
ax[1][0].set_title('Aligned')
ax[2][0].imshow(tile_3.FalseColor(bands))
ax[2][0].set_title('Unaligned mean filter')
ax[3][0].imshow(tile_4.FalseColor(bands))
ax[3][0].set_title('Aligned mean filter')

im = ax[0][1].imshow(sil_img1,cmap=cmap,vmin=vmin,vmax=vmax)
ax[0][1].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img1),fontsize=10)
im = ax[1][1].imshow(sil_img2,cmap=cmap,vmin=vmin,vmax=vmax)
ax[1][1].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img2),fontsize=10)
im = ax[2][1].imshow(sil_img3,cmap=cmap,vmin=vmin,vmax=vmax)
ax[2][1].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img3),fontsize=10)
im = ax[3][1].imshow(sil_img4,cmap=cmap,vmin=vmin,vmax=vmax)
ax[3][1].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img4),fontsize=10)

for i in range(4):
    for j in range(2):
        ax[i][j].axis('off')

# one shared colorbar
plt.colorbar(im, ax=ax.ravel().tolist(), shrink=0.2,location='top')

#plt.tight_layout()
plt.show()

### Exporting HAADF for additional annotation

In [ ]:
import tifffile as tf
tf.imwrite('/Users/aj/Desktop/haadf_00.tiff', Normalize_uint8(1-binning_xy(haadf_stack[0,:,:])))
tf.imwrite('/Users/aj/Desktop/haadf_19.tiff', Normalize_uint8(1-binning_xy(haadf_stack[19,:,:])))
tf.imwrite('/Users/aj/Desktop/haadf_99.tiff', Normalize_uint8(1-binning_xy(haadf_stack[99,:,:])))